# TDBP_2D file summary

This code processes and visualizes Synthetic Aperture Radar (SAR) data to create tomographic images, specifically using a technique known as Time-Domain Back-Projection (TDBP). The process involves several key steps:

1. **Gradient Calculation and Visualization:**

- The gradient of a Digital Terrain Model (DTM) MetaSensingDTM is computed.
- The magnitude of the gradient is plotted, providing a visual representation of terrain slope.
- The difference between a tomographic DTM (TomoDTMsr) and the MetaSensingDTM is also plotted.

2. **Preparation and Extraction of Data:**

- The code extracts specific azimuth indices and calculates various focusing centers for both the master and tomography datasets.
- It calculates the reference ranges (R_ref) for all tomography points and the master.

3. **Vertical Section Computation:**

- The code computes a vertical section of the data by integrating over different heights (dz_ax).
- It sums the squared magnitude of the interferometric data to generate an initial vertical section image.

4. **Coregistration and Geocoding:**

- The code coregisters the data to account for phase differences.
- Optionally, it performs geocoding based on the selected terrain model (geo_mod).

5. **Backprojection and Final Tomographic Image:**

- It performs backprojection, which involves computing the tomographic image by integrating data along curved paths corresponding to the radar signal travel times.
- The final tomographic image is plotted, showing the vertical section in ground coordinates.

In [ ]:
#import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [ ]:
#gradient calculation and initial plots
"""
This cell computes the gradient of MetaSensingDTM and plots the gradient
magnitude and the difference between TomoDTMsr and MetaSensingDTM.
"""

# Assuming MetaSensingDTM is defined

# Compute the gradient
a, b = np.gradient(MetaSensingDTM)

# Plot the gradient magnitude
plt.figure()
plt.imshow(np.sqrt(a**2 + b**2))
plt.colorbar()
plt.title('Gradient Magnitude')

# Plot the difference between TomoDTMsr and MetaSensingDTM
plt.figure()
plt.imshow(TomoDTMsr - MetaSensingDTM)
plt.colorbar()
plt.title('Difference between TomoDTMsr and MetaSensingDTM')

In [ ]:
#size and specific index setup
"""
This cell determines the size of the gradient arrays,
sets the azimuth index, and plots vertical lines on the previously generated plots.
"""
# Get the size of the gradient arrays
Ny, Nx = a.shape

# Set azimuth index
az_ind = 5458
# az_ind = 5484

# Plot vertical line on the gradient magnitude plot
plt.figure(1)
plt.imshow(np.sqrt(a**2 + b**2))
plt.plot([az_ind, az_ind], [0, Ny-1], 'r')
plt.colorbar()

# Plot vertical line on the difference plot
plt.figure(2)
plt.imshow(TomoDTMsr - MetaSensingDTM)
plt.plot([az_ind, az_ind], [0, Ny-1], 'r')
plt.colorbar()

In [ ]:
#extracting master and focusing centers
"""
This cell extracts the necessary focusing centers for the master from the provided data.
"""
# Assuming kz and FocusingCenters are defined

# Sum along specific axes
aaa = np.sum(np.sum(np.abs(kz), axis=1), axis=0)
Master = np.where(aaa == 0)[0]

# Extract focusing centers for the master
Sy_tx_Master = FocusingCenters[1, az_ind, Master, 0]
Sy_rx_Master = FocusingCenters[1, az_ind, Master, 1]
Sz_tx_Master = FocusingCenters[2, az_ind, Master, 0]
Sz_rx_Master = FocusingCenters[2, az_ind, Master, 1]

In [ ]:
#focusing centres for tomography and range reference calculation
"""
This cell calculates the range references needed for further processing.
"""
# Assuming n_tomo and ground_range_axis are defined

N = np.sum(n_tomo)
Sy_tx = FocusingCenters[1, az_ind, n_tomo, 0]
Sy_rx = FocusingCenters[1, az_ind, n_tomo, 1]
Sz_tx = FocusingCenters[2, az_ind, n_tomo, 0]
Sz_rx = FocusingCenters[2, az_ind, n_tomo, 1]

y_ax = ground_range_axis
Z_ref = MetaSensingDTM[:, az_ind]
z_dtm = TomoDTMsr[:, az_ind]
R_ref = np.zeros((len(y_ax), N))

for n in range(N):
    R_tx = np.sqrt((Sy_tx[n] - y_ax)**2 + (Sz_tx[n] - Z_ref)**2)
    R_rx = np.sqrt((Sy_rx[n] - y_ax)**2 + (Sz_rx[n] - Z_ref)**2)
    R_ref[:, n] = (R_tx + R_rx) / 2

R_tx_Master = np.sqrt((Sy_tx_Master - y_ax)**2 + (Sz_tx_Master - Z_ref)**2)
R_rx_Master = np.sqrt((Sy_rx_Master - y_ax)**2 + (Sz_rx_Master - Z_ref)**2)
R_ref_Master = (R_tx_Master + R_rx_Master) / 2

In [ ]:
#vertical section calculation
"""
This cell calculates the range references needed for further processing.
"""

# Assuming I and lambda are defined

dz_ax = np.arange(-30, 71, 1)
z_axis = dz_ax
Nz = len(z_axis)
Ngr = len(y_ax)
S = np.zeros((Nz, Ngr))
F = np.sum(np.abs(I[:, az_ind, n_tomo])**2, axis=2)

for z_ind in range(Nz):
    z = z_axis[z_ind]
    Igs = I[:, az_ind, n_tomo] * np.exp(-1j * kz[:, az_ind, n_tomo] * (TomoDTMsr[:, az_ind] - MetaSensingDTM[:, az_ind]))
    S[z_ind, :] = (np.abs(np.sum(Igs * np.exp(-1j * kz[:, az_ind, n_tomo] * z), axis=2))**2 / (F + np.finfo(float).eps)).T

plt.figure()
plt.subplot(2, 1, 1)
plt.imshow(S, extent=[0, len(y_ax), dz_ax[0], dz_ax[-1]], aspect='auto')
plt.colorbar()
plt.plot(np.arange(len(y_ax)), np.zeros_like(y_ax), 'w')
plt.title('Vertical section')
plt.xlabel('ground range [pix]')
plt.ylabel('height above the ground [m]')

In [ ]:
#data coregistration and geocoding
"""
This cell performs data coregistration and optionally geocodes the data based on the provided parameters.
"""
phi_dtm = kz[:, az_ind, n_tomo] * np.outer(z_dtm - Z_ref, np.ones(N))
geo_mod = 0 # 0 <=> geocode on Mauro's dtm; 1 <=> geocode on MS dtm

if geo_mod:
    data_gs_ref = I[:, az_ind, n_tomo] * np.exp(-1j * phi_dtm)
else:
    data_gs_ref = I[:, az_ind, n_tomo]

data_coreg = data_gs_ref * np.exp(1j * 4 * np.pi / lambda * R_ref)

Sy_Master = (Sy_tx_Master + Sy_rx_Master) / 2
Sz_Master = (Sz_tx_Master + Sz_rx_Master) / 2
y_of_R = Sy_Master + np.sqrt(R_ref_Master**2 - (Sz_Master - z_dtm)**2)
z_dtm_ground = np.interp(y_ax, y_of_R, z_dtm, left=0, right=0)

if geo_mod:
    z_dtm_ground = Z_ref

Z, Y = np.meshgrid(dz_ax, y_ax, indexing='ij')
Z += np.tile(z_dtm_ground, (len(dz_ax), 1))

In [ ]:
#backprojection and final plot
"""
This cell performs the backprojection and generates the final vertical section plot.
"""
Tomo_tdbp = 0
AAA = 0

for n in range(N):
    R_tx = np.sqrt((Sy_tx[n] - Y)**2 + (Sz_tx[n] - Z)**2)
    R_rx = np.sqrt((Sy_rx[n] - Y)**2 + (Sz_rx[n] - Z)**2)
    R = (R_tx + R_rx) / 2
    ddd = np.interp(R, R_ref[:, n], data_coreg[:, n]) * np.exp(-1j * 4 * np.pi / lambda * R)
    Tomo_tdbp += ddd
    AAA += np.abs(ddd)**2

plt.subplot(2, 1, 2)
plt.imshow(np.abs(Tomo_tdbp)**2 / AAA, extent=[0, len(y_ax), dz_ax[0], dz_ax[-1]], aspect='auto')
plt.colorbar()
plt.plot(np.arange(len(y_ax)), np.zeros_like(y_ax), 'w')
plt.title('Vertical section in ground coordinates by TDBP')
plt.xlabel('ground range [pix]')
plt.ylabel('height above the ground [m]')
plt.show()